# Peninjauan Kualitas Data
**Data pengguna, produk, dan transaksi**

Dokumen kerja ini digunakan untuk memeriksa data masuk, menguji fungsi normalisasi, dan meninjau hasil pipeline. Implementasi tetap ditempatkan di folder `src`.


## 1. Pemeriksaan lingkungan
Notebook harus dijalankan dari root project, yaitu folder yang berisi `src`, `data`, dan `quality_checks`.


In [1]:
from pathlib import Path
import sys, json, importlib, subprocess

print("Python:", sys.version.split()[0])
print("Working directory:", Path.cwd())
assert Path("src/cleaners.py").exists(), "Buka notebook dari root project"
print("Environment: READY")


Python: 3.12.0
Working directory: d:\Tutorial\Data Engineer\FSB_Data_Quality_Instructor_Pack\notebooks


AssertionError: Buka notebook dari root project

## 2. Peninjauan data masuk
Catat jumlah record, tipe container, dan satu contoh record. Pemeriksaan ini dilakukan sebelum menetapkan aturan cleaning.


In [ ]:
from src.io_utils import read_json

users_raw = read_json("data/raw/users.json")
products_raw = read_json("data/raw/products.json")
transactions_raw = read_json("data/raw/transactions.json")

for name, rows in [
    ("users", users_raw),
    ("products", products_raw),
    ("transactions", transactions_raw),
]:
    print(f"{name}: container={type(rows).__name__}, records={len(rows)}, record={type(rows[0]).__name__}")
    print("sample:", rows[0])
    print()


**Catatan peninjauan**

Isi `workpapers/RECORD_REVIEW.md`. Identifikasi field wajib, nilai yang perlu dinormalisasi, dan kondisi yang menyebabkan record ditolak.


## 3. Fungsi Python dasar
Implementasi berada di `src/foundations.py`. Kolom `actual` harus sama dengan `expected`.


In [ ]:
import src.foundations as foundations
importlib.reload(foundations)

def review_case(label, fn, args, expected):
    try:
        actual = fn(*args)
        status = "PASS" if actual == expected else "REVIEW"
    except Exception as exc:
        actual = f"{type(exc).__name__}: {exc}"
        status = "REVIEW"
    print(f"{label}\n  actual:   {actual!r}\n  expected: {expected!r}\n  status:   {status}\n")

review_case("clean_name", foundations.clean_name, ("  rani  ",), "Rani")
review_case("safe_city", foundations.safe_city, ({"name": "Rani"},), "Unknown")
review_case("classify_quantity", foundations.classify_quantity, ("12",), "numeric-string")
review_case("numbered_names", foundations.numbered_names, (["Rani", "Bima"],), [(1, "Rani"), (2, "Bima")])
review_case("parse_quantity", foundations.parse_quantity, ("two",), None)


**Pemeriksaan melalui terminal**
```powershell
python quality_checks_python_core.py
```
Hasil yang diharapkan: `8/8 core cases passed`.


## 4. Kontrak fungsi bantu
Helper functions must return stable values for missing, malformed, and valid input. Implementasi berada di `src/cleaners.py`.


In [ ]:
import src.cleaners as cleaners
importlib.reload(cleaners)

helper_cases = [
    ("normalize_text", cleaners.normalize_text, ("  Rani  ",), "Rani"),
    ("normalize_text empty", cleaners.normalize_text, ("   ",), None),
    ("normalize_email", cleaners.normalize_email, (" RANI@MAIL.COM ",), "rani@mail.com"),
    ("normalize_email invalid", cleaners.normalize_email, ("rani-at-mail.com",), None),
    ("parse_non_negative_float", cleaners.parse_non_negative_float, ("225000.50",), 225000.5),
    ("parse_positive_int", cleaners.parse_positive_int, ("2",), 2),
    ("parse_positive_int decimal", cleaners.parse_positive_int, (2.5,), None),
]

for label, fn, args, expected in helper_cases:
    review_case(label, fn, args, expected)


**Pemeriksaan melalui terminal**
```powershell
python quality_checks_helper_contracts.py
```
Hasil yang diharapkan: `11/11 helper cases passed`.


## 5. Pemrosesan data pengguna
Setiap record menghasilkan salah satu dari tiga outcome:
- masuk ke clean output;
- masuk ke rejected output dengan alasan;
- menghasilkan entry pada decision log jika terjadi `DROP`, `FIX`, atau `NULL`.


In [ ]:
importlib.reload(cleaners)
try:
    users_clean, users_rejected, user_decisions = cleaners.clean_users(users_raw)
    print(f"users: raw={len(users_raw)} clean={len(users_clean)} rejected={len(users_rejected)}")
    print("clean sample:", users_clean[:2])
    print("rejected:", users_rejected)
    print("decision sample:", user_decisions[:5])
except Exception as exc:
    print(f"PERLU DITINJAU: {type(exc).__name__}: {exc}")


Hasil yang diharapkan: `raw=16 clean=14 rejected=2`.

Tinjau sekurang-kurangnya satu record untuk setiap action yang digunakan: `DROP`, `FIX`, and `NULL`.


## 6. Pemrosesan data produk dan transaksi
Apply the same control pattern to the remaining entities. Transaction reference validation is outside the scope of this stage.


In [ ]:
importlib.reload(cleaners)
for name, rows, fn in [
    ("products", products_raw, cleaners.clean_products),
    ("transactions", transactions_raw, cleaners.clean_transactions),
]:
    try:
        clean, rejected, decisions = fn(rows)
        print(f"{name}: raw={len(rows)} clean={len(clean)} rejected={len(rejected)} decisions={len(decisions)}")
    except Exception as exc:
        print(f"{name}: PERLU DITINJAU — {type(exc).__name__}: {exc}")


Hasil yang diharapkan:
- products: `raw=12 clean=10 rejected=2`
- transactions: `raw=15 clean=12 rejected=3`


## 7. Pembuatan dan validasi output
Run the project from a clean process. This step writes JSON outputs and verifies counts, identifiers, rejection reasons, and decision-log schema.


In [ ]:
for command in [
    [sys.executable, "run_pipeline.py"],
    [sys.executable, "validate_delivery.py"],
]:
    result = subprocess.run(command, text=True, capture_output=True)
    print("$", " ".join(command))
    print(result.stdout)
    if result.stderr:
        print(result.stderr)


## 8. Peninjauan hasil
Complete `workpapers/DELIVERY_REVIEW.md` and confirm that `validate_delivery.py` returns `READY FOR REVIEW`.
